# Week 5, Lab 4 — Same task, two frameworks


In [ ]:
WEEK = 'Week 5'
LAB = 'Lab 4 — comparison'

import sys
from pathlib import Path

def _course_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "shared" / "course_runtime.py").exists():
            return p
    for c in [
        here / "agentic_ai_local",
        Path("/content/agentic_ai_local"),
        Path("/content"),
    ]:
        if (c / "shared" / "course_runtime.py").exists():
            return c
    return here

ROOT = _course_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shared.course_runtime import (
    detect_backend,
    print_banner,
    local_chat,
    calculator,
    lookup_fact,
    today_date,
    extract_json_object,
    parse_tool_call,
    openai_client_kwargs,
    get_langchain_llm,
    TOOL_SCHEMAS,
    MOCK_KB,
)

BACKEND = print_banner(WEEK, LAB)
print("If import failed, unzip/clone the WHOLE course folder (not a single notebook).")


In [ ]:
if BACKEND == "huggingface":
    %pip install -q transformers torch accelerate fastapi uvicorn pyautogen pydantic-ai openai
else:
    %pip install -q pyautogen pydantic-ai ollama openai


In [ ]:
print(calculator("45*12+30"))
print(lookup_fact("langgraph"))

cfg = openai_client_kwargs()
llm_config = {"config_list": [{
    "model": cfg["model"], "base_url": cfg["base_url"], "api_key": cfg["api_key"], "price": [0, 0],
}], "temperature": 0.1}

import autogen

def calc(expression: str) -> str:
    return calculator(expression)

def fact(topic: str) -> str:
    return lookup_fact(topic)

assistant = autogen.AssistantAgent(
    "assistant",
    llm_config={**llm_config, "functions": [
        {"name": "calc", "description": "arithmetic", "parameters": {"type": "object", "properties": {"expression": {"type": "string"}}, "required": ["expression"]}},
        {"name": "fact", "description": "local kb", "parameters": {"type": "object", "properties": {"topic": {"type": "string"}}, "required": ["topic"]}},
    ]},
    system_message="Call calc or fact. Then answer. Then TERMINATE.",
)
user = autogen.UserProxyAgent(
    "user",
    human_input_mode="NEVER",
    code_execution_config=False,
    max_consecutive_auto_reply=4,
    function_map={"calc": calc, "fact": fact},
    is_termination_msg=lambda m: m.get("content") and "TERMINATE" in str(m.get("content")),
)
print("==== AutoGen ====")
user.initiate_chat(assistant, message="What is 45*12+30?")


In [ ]:
from pydantic import BaseModel
from pydantic_ai import Agent

class Route(BaseModel):
    kind: str
    expression: str | None = None
    topic: str | None = None

try:
    from pydantic_ai.models.openai import OpenAIChatModel
    from pydantic_ai.providers.openai import OpenAIProvider
    model = OpenAIChatModel(cfg["model"], provider=OpenAIProvider(base_url=cfg["base_url"], api_key=cfg["api_key"]))
except Exception:
    from pydantic_ai.models.openai import OpenAIModel
    model = OpenAIModel(cfg["model"], base_url=cfg["base_url"], api_key=cfg["api_key"])

router = Agent(model, output_type=Route, instructions="Classify the user question. kind is math or research.")
route = router.run_sync("What is LangGraph?").output
print("==== Pydantic AI route ====", route)
if route.kind == "math":
    print("tool:", calculator(route.expression or "0"))
else:
    print("tool:", lookup_fact(route.topic or "langgraph"))


## Fill this table

| | Week 1 loop | LangGraph | AutoGen | Pydantic AI | CrewAI |
|---|---|---|---|---|---|
| Control flow | | | | | |
| Tools | | | | | |
| Best when | | | | | |
